In [5]:
! pip install pandas

In [7]:
import pandas as pd

In [ ]:
# 1. Load Excel

In [9]:
file_path = 'sample_db.xlsx'

In [11]:
required_sheets = [
    'database', 'megatrends', 'indicators', 'municipalities',
    'donors', 'donorgroups', 'stakeholders', 'beneficiary'
]

In [15]:
available_sheets = pd.ExcelFile(file_path).sheet_names
missing_sheets = [sheet for sheet in required_sheets if sheet not in available_sheets]

In [106]:
if missing_sheets:
    raise ValueError(f"missing sheets: {', '.join(missing_sheets)}")
else:
    print(f"load successfully: {', '.join(required_sheets)}")

load successfully: database, megatrends, indicators, municipalities, donors, donorgroups, stakeholders, beneficiary


In [27]:
# Load sheets
db = pd.read_excel(file_path, sheet_name='database')
megatrends = pd.read_excel(file_path, sheet_name='megatrends')
indicators = pd.read_excel(file_path, sheet_name='indicators')
municipalities = pd.read_excel(file_path, sheet_name='municipalities')
donors = pd.read_excel(file_path, sheet_name='donors')
donorgroups = pd.read_excel(file_path, sheet_name='donorgroups')
stakeholders = pd.read_excel(file_path, sheet_name='stakeholders')
beneficiaries = pd.read_excel(file_path, sheet_name='beneficiary')

In [29]:
# 2. Merge Dimensions Table

In [31]:
# merge indicators
db = db.merge(indicators[['id', 'name', 'group', 'beneficiaries_specific', 'stakeholders_specific', 'methods_specific']],
              left_on='indicator_id', right_on='id', how='left')
db.rename(columns={
    'name': 'indicator_name',
    'group': 'megatrend'
}, inplace=True)
db.drop(columns=['id_x', 'id_y', 'id'], errors='ignore', inplace=True)

In [94]:
# merge municipalities
db = db.merge(municipalities[['id', 'name']], left_on='city_ids', right_on='id', how='left')
db.rename(columns={'name': 'municipality_name'}, inplace=True)
db.drop(columns=['id'], inplace=True)
# Ensure municipality_name exists before final_columns selection
if 'municipality_name' not in db.columns:
    db['municipality_name'] = None

In [96]:
print("Merge indicators and municipalities.")

Merge indicators and municipalities.


In [ ]:
# 3. Prepare Dates and KPI Values

In [41]:
db['finalization_date'] = pd.to_datetime(db['finalization_date'], errors='coerce')
db['finalization_date'] = db['finalization_date'].dt.strftime('%Y-%m-%d')
db['year'] = pd.to_datetime(db['finalization_date'], errors='coerce').dt.year

In [43]:
db['value'] = db['baseline_value']

In [45]:
print("Prepare dates and KPI values.")

Prepare dates and KPI values.


In [47]:
# 4. Explode Multi-Value Fields

In [61]:
db = db.drop(columns=['municipality_name'], errors='ignore')

In [66]:
def explode_column(df, column, sep=';'):
    return df.assign(**{column: df[column].astype(str).str.split(sep)}).explode(column)

In [68]:
db['city_ids'] = db['city_ids'].astype(str)
db['donor_ids'] = db['donor_ids'].astype(str)
db['stakeholder_ids'] = db['stakeholder_ids'].astype(str)

In [70]:
db = explode_column(db, 'city_ids')
db = explode_column(db, 'donor_ids')
db = explode_column(db, 'stakeholder_ids')

In [72]:
for col in ['city_ids', 'donor_ids', 'stakeholder_ids']:
    db[col] = db[col].str.strip()
    db[col] = pd.to_numeric(db[col], errors='coerce')

In [74]:
print("Explod multi-value fields.")

Explod multi-value fields.


In [76]:
# 5. Merge Readable Names

In [78]:
db = db.merge(donors[['id', 'name', 'group_id']], left_on='donor_ids', right_on='id', how='left')
db.rename(columns={'name': 'donor_name'}, inplace=True)
db.drop(columns=['id'], inplace=True)

db = db.merge(donorgroups[['id', 'name']], left_on='group_id', right_on='id', how='left')
db.rename(columns={'name': 'donorgroup_name'}, inplace=True)
db.drop(columns=['id', 'group_id'], inplace=True)

In [80]:
db = db.merge(stakeholders[['id', 'name', 'group', 'category']], left_on='stakeholder_ids', right_on='id', how='left')
db.rename(columns={
    'name': 'stakeholder_name',
    'group': 'stakeholder_group',
    'category': 'stakeholder_category'
}, inplace=True)
db.drop(columns=['id'], inplace=True)

In [82]:
db = db.merge(beneficiaries[['id', 'name', 'group']], left_on='beneficiary_id', right_on='id', how='left')
db.rename(columns={
    'name': 'beneficiary_name',
    'group': 'beneficiary_group'
}, inplace=True)
db.drop(columns=['id'], inplace=True)

In [84]:
print("Merge donor, donor group, stakeholder, and beneficiary names.")

Merge donor, donor group, stakeholder, and beneficiary names.


In [ ]:
# 6. Save Output

In [98]:
final_columns = [
    'finalization_date', 'year', 'note', 'value', 'target_value',
    'indicator_name', 'megatrend', 'beneficiaries_specific', 'stakeholders_specific', 'methods_specific',
    'municipality_name', 'donor_name', 'donorgroup_name',
    'stakeholder_name', 'stakeholder_group', 'stakeholder_category',
    'beneficiary_name', 'beneficiary_group'
]
db = db[final_columns]

In [102]:
excel_output_path = 'giz_data_cogno_load.xlsx'
db.to_excel(excel_output_path, index=False)
print(f"save as Excel: {excel_output_path}")

save as Excel: giz_data_cogno_load.xlsx


In [104]:
csv_output_path = 'giz_data_cogno_load.csv'
db.to_csv(csv_output_path, index=False)
print(f"saved as CSV: {csv_output_path}")

saved as CSV: giz_data_cogno_load.csv
